# ISLES'24 NCCT-only nnU-Net v2 five-fold baseline

This notebook continues from the completed Dataset504 staging in Google Drive.

Run **Runtime → Run all** with a GPU runtime. It will:

- mount Drive;
- load the latest repository snapshot;
- install the pinned `nnunetv2==2.8.1` baseline environment;
- re-run staging in resume mode so labels are header-canonicalized to NCCT without voxel resampling;
- plan and preprocess `3d_fullres` once, with nnU-Net dataset integrity verification;
- restore our deterministic five-fold patient split;
- train folds 0–4 sequentially with `--npz`;
- skip completed folds and resume interrupted folds from nnU-Net checkpoints;
- persist preprocessing, checkpoints, validation outputs, logs and status JSON in Google Drive.

If Colab disconnects, reopen this notebook and **Run all again**. nnU-Net checkpoints every 50 epochs and the orchestrator uses `--c` to continue interrupted training.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Persistent paths and GPU check


In [ ]:
from pathlib import Path
import os, shutil, subprocess

ROOT = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024')
SOURCE = ROOT / 'ncct_source'
NNUNET_RAW = ROOT / 'nnUNet_raw'
NNUNET_PREPROCESSED = ROOT / 'nnUNet_preprocessed'
NNUNET_RESULTS = ROOT / 'nnUNet_results'
RESULTS = ROOT / 'results'
LOG = RESULTS / 'nnunet_ncct_cv.log'
STATUS = RESULTS / 'nnunet_ncct_cv_status.json'
REPO_ZIP = Path('/content/HybridStrokeSeg-main.zip')
REPO = Path('/content/HybridStrokeSeg-Reproduction-main')

for path in (NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS, RESULTS):
    path.mkdir(parents=True, exist_ok=True)

os.environ['nnUNet_raw'] = str(NNUNET_RAW)
os.environ['nnUNet_preprocessed'] = str(NNUNET_PREPROCESSED)
os.environ['nnUNet_results'] = str(NNUNET_RESULTS)

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('A GPU runtime is required. In Colab select Runtime > Change runtime type > GPU.')
print('GPU:', torch.cuda.get_device_name(0))


## Load the latest repository and install the pinned modern baseline


In [ ]:
def run(cmd, *, cwd=None):
    cmd = [str(x) for x in cmd]
    print('$', ' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True, cwd=cwd)

if REPO.exists():
    shutil.rmtree(REPO)
if REPO_ZIP.exists():
    REPO_ZIP.unlink()

run([
    'curl', '-fL', '--retry', '20', '--retry-delay', '3', '--retry-all-errors',
    'https://github.com/NTinkicht/HybridStrokeSeg-Reproduction/archive/refs/heads/main.zip',
    '-o', REPO_ZIP,
])
run(['unzip', '-q', REPO_ZIP, '-d', '/content'])
os.chdir(REPO)
run(['python', '-m', 'pip', 'install', '-q', '-e', '.[modern]'])
run(['python', '-c', 'import nnunetv2; print("nnU-Net v2 import OK")'])


## Re-stage Dataset504 in resume mode

This is fast because existing images are kept. It also canonicalizes staged label headers to the corresponding NCCT geometry **without resampling any voxel values**, preventing benign qform/sform rounding from failing nnU-Net's own integrity checker.


In [ ]:
run([
    'python', 'scripts/stage_isles2024_nnunet.py', SOURCE, NNUNET_RAW,
    '--dataset-id', '504',
    '--dataset-name', 'ISLES2024_NCCT',
    '--channels', 'ncct',
    '--mode', 'copy',
    '--resume',
    '--split-seed', '2026',
], cwd=REPO)


## Plan, preprocess and train all five folds

The command below is intentionally resumable. Completed preprocessing and completed folds are skipped; an interrupted fold resumes from its latest checkpoint.


In [ ]:
def run_logged(cmd):
    cmd = [str(x) for x in cmd]
    print('$', ' '.join(cmd), flush=True)
    with LOG.open('a', encoding='utf-8') as log_handle:
        log_handle.write('\n$ ' + ' '.join(cmd) + '\n')
        log_handle.flush()
        proc = subprocess.Popen(
            cmd,
            cwd=REPO,
            env=os.environ.copy(),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_handle.write(line)
            log_handle.flush()
        code = proc.wait()
        if code != 0:
            raise subprocess.CalledProcessError(code, cmd)

run_logged([
    'python', 'scripts/run_isles2024_nnunet_cv.py',
    '--dataset-id', '504',
    '--configuration', '3d_fullres',
    '--folds', '0', '1', '2', '3', '4',
    '--status-json', STATUS,
])


## Persistent status


In [ ]:
import json

print('Persistent log:', LOG)
print('Persistent status:', STATUS)
if STATUS.exists():
    print(json.dumps(json.loads(STATUS.read_text()), indent=2))
print('nnUNet_preprocessed:', NNUNET_PREPROCESSED)
print('nnUNet_results:', NNUNET_RESULTS)


## Reconnection rule

If the runtime disconnects during preprocessing or training, reopen this same notebook with a GPU runtime and choose **Runtime → Run all**. Do not delete `nnUNet_preprocessed`, `nnUNet_results`, or the status/log files in Drive.
